# Notebook 02: Naive Modeling and Two-Model Baselines

This notebook intentionally starts with a wrong causal framing (naive treated-response ranking)
and then moves to a better uplift-aware baseline (two-model approach).

## Non-negotiables for fair comparison
- same preprocessing pipeline as Notebook 01
- same train/validation/test split strategy
- same seed (`42`)
- same test set for all baseline comparisons
- primary target = `conversion`

In [ ]:
from pathlib import Path
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='talk')

if Path.cwd().name == 'notebooks':
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import set_seed, ensure_output_dirs, save_plot
from src.preprocessing import (
    load_hillstrom_data,
    basic_cleaning,
    split_features_outcomes_treatment,
    create_train_valid_test_split,
)
from src.baselines import (
    fit_naive_treated_response_model,
    predict_naive_treated_response_model,
    fit_two_model_uplift,
    predict_two_model_uplift,
)
from src.evaluation import (
    qini_curve, qini_coefficient, auuc_score, uplift_curve,
    cumulative_gain_curve, uplift_by_decile, treatment_control_counts_by_decile,
)

SEED = 42
set_seed(SEED)
OUT_DIRS = ensure_output_dirs(PROJECT_ROOT)
DATA_PATH = PROJECT_ROOT / 'data' / 'hillstrom.csv'

In [ ]:
df = basic_cleaning(load_hillstrom_data(str(DATA_PATH)))
X, y, t = split_features_outcomes_treatment(df, outcome_col='conversion', treatment_col='treatment')
splits = create_train_valid_test_split(X, y, t, test_size=0.2, valid_size=0.2, random_state=SEED, stratify=True)

print('X shape:', X.shape)
print('Train/Valid/Test sizes:', len(splits['X_train']), len(splits['X_valid']), len(splits['X_test']))
display(pd.Series({k: len(v) for k, v in splits.items() if k.startswith('X_')}))

## 1) Naive treated-response model
This model learns `P(conversion | treated, X)` and ranks by conversion propensity among treated users.
This often prioritizes likely buyers (including sure things), not necessarily persuadables.

In [ ]:
naive_model = fit_naive_treated_response_model(
    splits['X_train'], splits['y_train'], splits['t_train'],
    model_family='xgboost', random_state=SEED
)
naive_scores_test = predict_naive_treated_response_model(naive_model, splits['X_test'])
print('Naive score summary:')
print(pd.Series(naive_scores_test).describe())

## 2) Two-model uplift baseline

In [ ]:
two_t, two_c = fit_two_model_uplift(
    splits['X_train'], splits['y_train'], splits['t_train'],
    model_family='xgboost', random_state=SEED
)
two_model_uplift_test = predict_two_model_uplift(two_t, two_c, splits['X_test'])
print('Two-model uplift score summary:')
print(pd.Series(two_model_uplift_test).describe())

## 3) Baseline evaluation on same holdout test set

In [ ]:
def compute_metrics(y_true, t_true, score):
    return {
        'qini': qini_coefficient(y_true, t_true, score),
        'auuc': auuc_score(y_true, t_true, score),
    }

metrics = pd.DataFrame([
    {'model': 'naive_treated_response', **compute_metrics(splits['y_test'], splits['t_test'], naive_scores_test)},
    {'model': 'two_model_uplift', **compute_metrics(splits['y_test'], splits['t_test'], two_model_uplift_test)},
]).sort_values('qini', ascending=False).reset_index(drop=True)
display(metrics)
metrics.to_csv(OUT_DIRS['tables'] / 'nb02_baseline_metrics.csv', index=False)

In [ ]:
curves = {
    'naive': qini_curve(splits['y_test'], splits['t_test'], naive_scores_test),
    'two_model': qini_curve(splits['y_test'], splits['t_test'], two_model_uplift_test),
}
fig, ax = plt.subplots(figsize=(10, 7))
for name, c in curves.items():
    ax.plot(c['fraction_targeted'], c['incremental_outcomes'], label=name)
ax.set_title('Qini Curves: Naive vs Two-Model')
ax.set_xlabel('Fraction targeted')
ax.set_ylabel('Cumulative incremental conversions')
ax.legend()
plt.tight_layout()
save_plot(fig, OUT_DIRS['figures'] / 'nb02_qini_baselines.png')
plt.show()

In [ ]:
uplift_curves = {
    'naive': uplift_curve(splits['y_test'], splits['t_test'], naive_scores_test),
    'two_model': uplift_curve(splits['y_test'], splits['t_test'], two_model_uplift_test),
}
fig, ax = plt.subplots(figsize=(10, 7))
for name, c in uplift_curves.items():
    ax.plot(c['fraction_targeted'], c['uplift'], label=name)
ax.set_title('Uplift Curves: Naive vs Two-Model')
ax.set_xlabel('Fraction targeted')
ax.set_ylabel('Estimated uplift')
ax.legend()
plt.tight_layout()
save_plot(fig, OUT_DIRS['figures'] / 'nb02_uplift_baselines.png')
plt.show()

In [ ]:
gain_curves = {
    'naive': cumulative_gain_curve(splits['y_test'], splits['t_test'], naive_scores_test),
    'two_model': cumulative_gain_curve(splits['y_test'], splits['t_test'], two_model_uplift_test),
}
fig, ax = plt.subplots(figsize=(10, 7))
for name, c in gain_curves.items():
    ax.plot(c['fraction_targeted'], c['cumulative_gain'], label=name)
ax.set_title('Cumulative Gain Curves: Naive vs Two-Model')
ax.set_xlabel('Fraction targeted')
ax.set_ylabel('Cumulative gain')
ax.legend()
plt.tight_layout()
save_plot(fig, OUT_DIRS['figures'] / 'nb02_cumulative_gain_baselines.png')
plt.show()

In [ ]:
decile_naive = uplift_by_decile(splits['y_test'], splits['t_test'], naive_scores_test).assign(model='naive')
decile_two = uplift_by_decile(splits['y_test'], splits['t_test'], two_model_uplift_test).assign(model='two_model')
count_naive = treatment_control_counts_by_decile(splits['y_test'], splits['t_test'], naive_scores_test).assign(model='naive')
count_two = treatment_control_counts_by_decile(splits['y_test'], splits['t_test'], two_model_uplift_test).assign(model='two_model')
deciles = pd.concat([decile_naive, decile_two], ignore_index=True)
counts = pd.concat([count_naive, count_two], ignore_index=True)
display(deciles.head(20))
deciles.to_csv(OUT_DIRS['tables'] / 'nb02_uplift_by_decile.csv', index=False)
counts.to_csv(OUT_DIRS['tables'] / 'nb02_treatment_control_counts_by_decile.csv', index=False)

## Interpretation
- Naive model ranks high propensity buyers, which can over-target sure things.
- Two-model uplift better approximates incremental ranking by subtracting control response.
- Two-model is still imperfect due to separate-model instability and error propagation.

## End-of-notebook summary
What failed: naive treated-response ranking for causal targeting.
What improved: two-model uplift better aligns with incremental decisioning.
Next: Notebook 03 compares S/T/X meta-learners with the same split and evaluation stack.